In [1]:
import os
import sys
import pickle
import numpy as np
from numba import njit
import itertools as itt
import aerosandbox as asb

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from Aircraft.Planform import Planform
from Aircraft.Fixed import Fixed
from Drag.Fuselage import Fuselage
from Drag.Bay import Bay
from Drag.LandingGear import LandingGear
from Aircraft.Aircraft import Aircraft
from global_parameters import Assumptions
from Requirements.FuelReq import FuelReq
from Requirements.LGReq import LGReq
from Requirements.MassReq import MassReq
from Requirements.MDReq import MDReq
from Requirements.EmpennageReq import EmpennageReq
from Requirements.Requirement import Requirement
from EmpennageSizing.TailFinder import TailFinder
from EmpennageSizing.CanardFinder import CanardFinder

from structural_analysis.Material import Material
from structural_analysis.iterative_planform_sizing import size_planform

In [2]:
assumptions = Assumptions()

#TODO load the fuselage here
with open("pickles/fixed_pickle.pcl", "rb") as f:
    fixed:Fixed = pickle.load(f)

for component in fixed.drag_components(False):
    component.add_cache_entry("go_around", assumptions.airspeed_approach/asb.Atmosphere(assumptions.altitude_go_round).speed_of_sound(), assumptions.altitude_go_round)
    component.add_cache_entry("mach_max", assumptions.mach_max, assumptions.altitude_mach_max)
    component.add_cache_entry("cruise", assumptions.mach_cruise, assumptions.altitude_cruise)
for component in fixed.drag_components(True):
    component.add_cache_entry("takeoff", assumptions.airspeed_approach/asb.Atmosphere().speed_of_sound(), 0.)

# Defining the standard aircraft with the standard planform
To be used when ppl don't wanna build their own planform

In [3]:
standard_wing = Planform(aspect_ratio=27, span=2.667, sweep_quarter_deg=15., taper=.5, thickness_to_chord=0.12, cm_quarter_chord=0,
                         wetted_surface_ratio=1.07, interference_factor=1.0, clmax=1.25, flap=False)

In [4]:
assumptions = Assumptions()

go_around_atmosphere = asb.Atmosphere(assumptions.altitude_go_round)
sea_level_atmosphere = asb.Atmosphere()

standard_wing.add_cache_entry('cruise', assumptions.mach_cruise, assumptions.altitude_cruise)
standard_wing.add_cache_entry('mach_max', assumptions.mach_max, assumptions.altitude_mach_max)
#NOTE: not fully technically correct but prevents coupling which would be problematic, acceptable as CD0 dept. on mach is small @ low mach
standard_wing.add_cache_entry('go_around', assumptions.airspeed_approach / go_around_atmosphere.speed_of_sound(), assumptions.altitude_go_round)
standard_wing.add_cache_entry('takeoff', assumptions.airspeed_approach / sea_level_atmosphere.speed_of_sound(), 0.)

In [5]:
material_skin = Material(assumptions.cfrp_density, elastic_modulus=assumptions.cfrp_Young_modulus, 
                         poisson_ratio=assumptions.cfrp_poisson, shear_modulus=assumptions.cfrp_Young_modulus / 2 / (1 + assumptions.cfrp_poisson),
                         yield_strength=assumptions.cfrp_yield_strength, fracture_strength=assumptions.cfrp_yield_strength)

fuselage_diameter = fixed.fuselage.diameter_max
size_planform(planform=standard_wing, thicknesses=assumptions.allowable_thicknesses, fuselage_diameter=fuselage_diameter, material_skin=material_skin, density_core=assumptions.foam_denisty)
    

77.21013976341868
Stresses 129458939.17300314, 1831224665.886364, 0.0004
77.21013976341868
Stresses 86752796.18752897, 1262809199.3369858, 0.0005959183673469389
77.21013976341868
Stresses 65179589.937134795, 976549460.245957, 0.0007918367346938775
77.21013976341868
Stresses 52164349.802599356, 804576017.409593, 0.0009877551020408164
77.21013976341868
Stresses 43457602.953979194, 690160949.2191758, 0.0011836734693877551
77.21013976341868
Stresses 37223778.2872156, 608800742.139436, 0.0013795918367346938
77.21013976341868
Stresses 32540334.884724833, 548180332.807061, 0.0015755102040816327
77.21013976341868
Stresses 28892860.529789634, 501432891.0668461, 0.0017714285714285716
77.21013976341868
Stresses 25971854.096169297, 464426942.4686692, 0.0019673469387755105
77.21013976341868
Stresses 23579935.62033685, 434527647.8370986, 0.002163265306122449
77.21013976341868
Stresses 21585290.80139003, 409975289.3365498, 0.002359183673469388
77.21013976341868
Stresses 19896534.00579607, 389549681.7

In [6]:
print(standard_wing.mass_cache, standard_wing.x_cg_cache)

0.47089862403897753 0.2171120555153211


# We consider the thing to be tailed

In [7]:
tail = TailFinder(fixed, material=material_skin, core_density=assumptions.foam_denisty, thicknesses=assumptions.allowable_thicknesses, safety_factor=assumptions.structural_safety_factor, AR_h=7., taper_h=.7, taper_v=.8).find_planforms(standard_wing)

print(tail[0].wing_area / standard_wing.wing_area)

0.2857716415267553
Stresses 828297.6600816031, 3875946.1406409834, 0.0004
0.28497438860667096
Stresses 3081977.82916016, 71293234.64530537, 0.0004
15.444532388825174
Stresses 16452606.0928979, 72582284.26488966, 0.0004
15.401551439952183
Stresses 8234662.707965811, 17993071.095320698, 0.0004
11.98540811627283
Stresses 13603534.451964324, 59989672.17989563, 0.0004
11.95205047598481
Stresses 7741919.527904089, 19232278.05150273, 0.0004
12.290895004290995
Stresses 13862734.410978103, 62270895.90056516, 0.0004
12.256687468488488
Stresses 7789551.609163478, 19456285.702903766, 0.0004
12.26136326099996
Stresses 13837748.199853215, 61297474.51517257, 0.0004
12.227237885234201
Stresses 7784986.962343926, 19202697.747402553, 0.0004
12.264193014691923
Stresses 13840143.04861774, 61019130.571263276, 0.0004
12.23005976628762
Stresses 7785424.716112311, 19111179.358717013, 0.0004
12.264057325032434
Stresses 13840028.216260094, 61053163.24402071, 0.0004
12.229924454129401
Stresses 7785403.727132165,

In [8]:
for t in tail:
    t.add_cache_entry('cruise', assumptions.mach_cruise, assumptions.altitude_cruise)
    t.add_cache_entry('mach_max', assumptions.mach_max, assumptions.altitude_mach_max)
#NOTE: not fully technically correct but prevents coupling which would be problematic, acceptable as CD0 dept. on mach is small @ low mach
    t.add_cache_entry('go_around', assumptions.airspeed_approach / go_around_atmosphere.speed_of_sound(), assumptions.altitude_go_round)
    t.add_cache_entry('takeoff', assumptions.airspeed_approach / sea_level_atmosphere.speed_of_sound(), 0.)

In [9]:
ac = Aircraft(fixed, [standard_wing] + tail)


In [10]:
for acp in ac.planforms:
    print(acp.mass_cache)

print(ac.fixed.x_cg_min, ac.fixed.x_cg_max, ac.fixed.x_LE_wing)

0.47089862403897753
0.013407581310887864
0.026059015056734715
1.031789359176587 1.09066819465422 1.125


# Requirement check for the aircraft

In [11]:
requirements:list[Requirement] = [
    MassReq(50.),
    MDReq(),
    FuelReq(),
    LGReq(),
    EmpennageReq(),
]

requirement_labels = [
    "MTOM",
    "Matching Diagram",
    "Fuel",
    "Landing Gear",
    "Empennage Requirement"
]

In [12]:
failed_reqs = list()
for requirement, label in zip(requirements, requirement_labels):
    if not requirement.assess(ac):
        failed_reqs.append(label)

if len(failed_reqs):
    print(f"ac mass: {ac.total_mass()}, {ac.planforms[0].oswald}")
    print(f"MainWing: AR={ac.planforms[0].aspect_ratio}, tc={ac.planforms[0].thickness_to_chord}, sweep={np.rad2deg(ac.planforms[0].sweep_quarter_rad)} deg, cmac={ac.planforms[0].cm_quarter_chord}")
    print(f"Failed: {failed_reqs}")
    print()

Fuel available: 11.000000001437998 kg
Fuel required: 5.263914430170283 kg
Difference: 5.736085571267715 kg
all constraints satisfied
ac mass: 34.574526226211546, 0.680337246179267
MainWing: AR=27, tc=0.12, sweep=14.999999999999998 deg, cmac=0
Failed: ['Empennage Requirement']



In [13]:
#TODO: ctrl surface sizing